In [1]:
from langchain.chat_models import ChatOpenAI
import os
API_SECRET_KEY = "sk-SYAZNbJfQ576lFyHM5FOoDeHCQRYy3cdntzYBeGHEzXsN2Ei"
BASE_URL = "https://api.fe8.cn/v1"
os.environ["OPENAI_API_KEY"] = API_SECRET_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL

chat = ChatOpenAI(temperature=0.0)
chat

C:\Users\tassa\AppData\Local\Temp\ipykernel_30404\3094856839.py:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI(temperature=0.0)


ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x000001C05C318100>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x000001C05D2F76D0>, temperature=0.0, model_kwargs={}, openai_api_key='sk-SYAZNbJfQ576lFyHM5FOoDeHCQRYy3cdntzYBeGHEzXsN2Ei', openai_api_base='https://api.fe8.cn/v1', openai_proxy='')

# 输出解析器

使用解析器，输出解析器本质就是个特殊的prompt模板，指定输出的json每个字段内容和取值范围

1. 输入对应格式提示词，比如需要一个json输出，那么提示词中要严格说明，让llm输出对应格式的json
2. llm输出对应格式的json
3. 提示词中最好引入CoT思维链，Thought，action，obervation

以下就是两个是否使用解析器的例子

## 例子 1. 不使用输出解析器的情况

从产品评论中提取我需要的对应格式信息

In [2]:
# 这是我需要的信息的一个例子
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

# 这是实际数据：用户评论，我要从里面抽出以上json内需要的信息
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

# 提示词模板：明确告诉llm，我需要抽取哪些信息
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

开始解析

In [ ]:
# 引入langchain字符串模板
from langchain.prompts import ChatPromptTemplate
llm_model = "gpt-3.5-turbo"
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)
# 格式化模板
messages = prompt_template.format_messages(text=customer_review)
# 创建模型，设置热度
chat = ChatOpenAI(temperature=0.0, model=llm_model)
# 调用模型，获得回复
response = chat(messages)
print(response.content)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


C:\Users\tassa\AppData\Local\Temp\ipykernel_30404\307671812.py:9: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chat(messages)


{
    "gift": true,
    "delivery_days": 2,
    "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}


In [ ]:
# 可以看到回复值是一个字符串，还不是一个pthon的dict，尤其是price_value不符合我们的输出形式，还需要从这一句话中提取关键数据
type(response.content)

str

## 例子2：使用输出解析器

In [10]:
# 使用输出解析器将llm的输出进行格式化
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

# 定义回复的所有属性约束，就是对每个字段给定说明，和取值
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

# 定义输出解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
# 获得格式化输出
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"gift": string  // Was the item purchased                             as a gift for someone else?                              Answer True if yes,                             False if not or unknown.
	"delivery_days": string  // How many days                                      did it take for the product                                      to arrive? If this                                       information is not found,                                      output -1.
	"price_value": string  // Extract any                                    sentences about the value or                                     price, and output them as a                                     comma separated Python list.
}
```


In [12]:
# 使用输出解析器的提示词
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```

In [13]:
# 得到了使用输出解析器的prompt，调用模型
response = chat(messages)

In [ ]:
print(response.content) # 查看模型内容，这时候有了```json .....```这样的结果，可以直接转为json格式
output_dict = output_parser.parse(response.content)  # 调用解析器转为python的数据格式

```json
{
	"gift": true,
	"delivery_days": 2,
	"price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}
```


In [16]:
output_dict

{'gift': True,
 'delivery_days': 2,
 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}